# lbmbench on Colab (A100): GPU-LBM review benchmark

This notebook runs the controlled benchmark of the review paper on a Colab GPU.

1. **Runtime → Change runtime type → A100 GPU** (needs Colab Pro / Pay-as-you-go).
2. Run the cells in order. Cell 3 asks you to upload `lbmbench_colab.zip`.
3. The last cell downloads `results_<gpu>.zip`. Send it to Claude for merging into the paper.

The full campaign takes roughly 20–40 min. Keep the tab open, because Colab disconnects idle sessions.
Each stage writes its CSVs as it goes, and you can save partial results to Google Drive (cell 9).

## 1. Check the GPU

In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap,memory.total,ecc.mode.current,power.limit --format=csv
!nvcc --version | tail -2

Sun Oct 18 11:03:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Detect the architecture flag

In [ ]:
import subprocess
cc = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
ARCH = "sm_" + cc.replace(".", "")
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
print("GPU:", gpu, "| compute capability", cc, "-> ARCH =", ARCH)
if "A100" not in gpu:
    print("WARNING: this is not an A100. The run still works, but check the runtime type.")

GPU: NVIDIA A100-SXM4-40GB | compute capability 8.0 -> ARCH = sm_80


## 3. Upload `lbmbench_colab.zip`

In [ ]:
from google.colab import files
import os, zipfile
os.chdir("/content")
up = files.upload()                      # choose lbmbench_colab.zip
name = [k for k in up if k.endswith(".zip")][0]
zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/lbmbench")
!ls -la

Saving lbmbench_colab.zip to lbmbench_colab.zip
total 80
drwxr-xr-x 2 root root  4096 Oct 18 11:03 .
drwxr-xr-x 1 root root  4096 Oct 18 11:03 ..
-rw-r--r-- 1 root root 12639 Oct 18 11:03 analyze.py
-rw-r--r-- 1 root root 39424 Oct 18 11:03 lbmbench.cu
-rw-r--r-- 1 root root   828 Oct 18 11:03 Makefile
-rw-r--r-- 1 root root  3122 Oct 18 11:03 README.md
-rw-r--r-- 1 root root  5256 Oct 18 11:03 run_all.sh


## 4. Build (compile only)

In [ ]:
!make clean && make ARCH={ARCH}
!grep -c "Compiling entry" ptxas_report.txt
!chmod +x run_all.sh lbmbench
!./lbmbench --mode info

rm -f lbmbench ptxas_report.txt
nvcc -O3 -std=c++17 --expt-relaxed-constexpr -lineinfo -arch=sm_80 -Xptxas -v lbmbench.cu -o lbmbench 2> ptxas_report.txt || (cat ptxas_report.txt; exit 1)
kernels compiled: 267
267
device,NVIDIA A100-SXM4-40GB
cc,8.0
sm_count,108
global_mem_GiB,39.49
l2_MiB,40.0
smem_per_block_KiB,48
regs_per_sm,65536
sm_clock_MHz,1410
mem_clock_MHz,1215
bus_bits,5120
theoretical_bw_GBs,1555.2
note: theoretical_bw assumes DDR factor 2; for GDDR7 check the vendor value.


## 5. Correctness check (about 1 min). Must end with `0` CHECK lines

In [ ]:
!./run_all.sh verify 2>&1 | tail -25

VERIFY: all patterns consistent
+ ./lbmbench --mode verify --q 27 --prec fp16c --coll bgk --n 24 --vsteps 51 --U 0.05 --csv results/cccae4a31ada_NVIDIA_A100-SXM4-40GB/verify.csv
verify abpull soa vs abpull/soa after 51 steps: max|diff| = 0.000e+00 (bitwise identical)
verify abpush soa vs abpull/soa after 51 steps: max|diff| = 0.000e+00 (bitwise identical)
verify aa     soa vs abpull/soa after 51 steps: max|diff| = 0.000e+00 (bitwise identical)
verify ep     soa vs abpull/soa after 51 steps: max|diff| = 0.000e+00 (bitwise identical)
reference mass drift per node: -3.905e-08
VERIFY: all patterns consistent
+ ./lbmbench --mode verify --q 27 --prec fp16c --coll trt --n 24 --vsteps 51 --U 0.05 --csv results/cccae4a31ada_NVIDIA_A100-SXM4-40GB/verify.csv
verify abpull soa vs abpull/soa after 51 steps: max|diff| = 0.000e+00 (bitwise identical)
verify abpush soa vs abpull/soa after 51 steps: max|diff| = 0.000e+00 (bitwise identical)
verify aa     soa vs abpull/soa after 51 steps: max|diff| = 0.

## 6. Performance experiments E1–E5 (longest stage)

In [ ]:
!./run_all.sh perf 2>&1 | grep -E "MLUPS|copy bandwidth|done" | tail -40

D3Q19 abpull soa fp32 bgk shift=1 N=256x256x256 : 8993.3 MLUPS, 152.0 B/cell/step, eff. BW 1367.0 GB/s, mem 152 B/cell
D3Q19 ep soa fp32 bgk shift=1 N=256x256x256 : 8736.8 MLUPS, 152.0 B/cell/step, eff. BW 1328.0 GB/s, mem 76 B/cell
D3Q19 ep soa fp16s bgk shift=1 N=256x256x256 : 16937.5 MLUPS, 76.0 B/cell/step, eff. BW 1287.2 GB/s, mem 38 B/cell
D3Q19 abpull soa fp32 bgk shift=1 N=320x320x320 : 8967.3 MLUPS, 152.0 B/cell/step, eff. BW 1363.0 GB/s, mem 152 B/cell
D3Q19 ep soa fp32 bgk shift=1 N=320x320x320 : 8631.1 MLUPS, 152.0 B/cell/step, eff. BW 1311.9 GB/s, mem 76 B/cell
D3Q19 ep soa fp16s bgk shift=1 N=320x320x320 : 16548.3 MLUPS, 76.0 B/cell/step, eff. BW 1257.7 GB/s, mem 38 B/cell
D3Q19 abpull soa fp32 bgk shift=1 N=384x384x384 : 8985.9 MLUPS, 152.0 B/cell/step, eff. BW 1365.9 GB/s, mem 152 B/cell
D3Q19 ep soa fp32 bgk shift=1 N=384x384x384 : 8672.2 MLUPS, 152.0 B/cell/step, eff. BW 1318.2 GB/s, mem 76 B/cell
D3Q19 ep soa fp16s bgk shift=1 N=384x384x384 : 16791.8 MLUPS, 76.0 B/ce

## 7. Accuracy (E6) and energy (E7)

In [ ]:
!./run_all.sh acc 2>&1 | grep -E "TGV|done" | tail -20
!./run_all.sh energy 2>&1 | grep -E "MLUPS|done"

TGV ep fp16s reg shift=1 L=128 U=0.02 tau=0.8 t=2075 : L2 = 1.0312e-02, mass drift = -1.421e-05
TGV ep fp16c bgk shift=1 L=128 U=0.02 tau=0.8 t=2075 : L2 = 8.2651e-04, mass drift = -4.794e-06
TGV ep fp16c reg shift=1 L=128 U=0.02 tau=0.8 t=2075 : L2 = 9.7336e-04, mass drift = -8.857e-06
TGV ep fp64 bgk shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 6.3917e-05, mass drift = -2.644e-17
TGV ep fp64 reg shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 5.5023e-05, mass drift = 9.744e-18
TGV ep fp32 bgk shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 1.8479e-04, mass drift = 7.714e-06
TGV ep fp32 reg shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 1.5115e-04, mass drift = 3.327e-06
TGV ep fp16s bgk shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 2.5769e-02, mass drift = -8.686e-06
TGV ep fp16s reg shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 4.0610e-02, mass drift = -9.347e-06
TGV ep fp16c bgk shift=1 L=256 U=0.01 tau=0.8 t=8300 : L2 = 9.6658e-03, mass drift = -2.782e-06
TGV ep fp16c reg shift=1 L=256 U=0.01 tau=0.8 t

## 8. Figures and tables for this GPU

In [ ]:
import glob
d = sorted(glob.glob("results/*"))[-1]
print("results dir:", d)
!python3 analyze.py {d} --fig figs --tab tabs
!cat tabs/bench_e2.tex tabs/bench_devices.tex
from IPython.display import IFrame

results dir: results/cccae4a31ada_NVIDIA_A100-SXM4-40GB
NVIDIA A100-SXM4-40GB: copy BW = 1357.6 GB/s, 381 perf rows, 120 acc rows
figures -> figs  tables -> tabs
\begin{tabular}{llrrrrr}
\toprule
GPU & Pattern & \multicolumn{5}{c}{D3Q19 BGK, MLUPS (\% of copy bandwidth)} \\
 & & FP64 & mixed & FP32 & FP16S & FP16C \\
\midrule
NVIDIA A100-SXM4-40GB & AB pull & 4512 (101) & 8950 (100) & 8992 (101) & 17266 (97) & 15135 (85) \\
NVIDIA A100-SXM4-40GB & AB push & 4399 (99) & 8796 (98) & 8755 (98) & 16649 (93) & 14660 (82) \\
NVIDIA A100-SXM4-40GB & AA & 4405 (99) & 8784 (98) & 8712 (98) & 16617 (93) & 14941 (84) \\
NVIDIA A100-SXM4-40GB & Esoteric pull & 4402 (99) & 8775 (98) & 8737 (98) & 16938 (95) & 14787 (83) \\
\bottomrule
\end{tabular}
\begin{tabular}{lrr}
\toprule
GPU & measured copy bandwidth (GB/s) & D3Q19 FP32 AB-pull roofline (MLUPS) \\
\midrule
NVIDIA A100-SXM4-40GB & 1358 & 8931 \\
\bottomrule
\end{tabular}


## 9. (Optional) Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/lbmbench_results && cp -r results figs tabs /content/drive/MyDrive/lbmbench_results/
print("saved to MyDrive/lbmbench_results")

Mounted at /content/drive
saved to MyDrive/lbmbench_results


## 10. Download results

In [ ]:
import shutil
tag = gpu.replace(" ", "_")
shutil.make_archive(f"/content/results_{tag}", "zip", "/content/lbmbench", ".")
files.download(f"/content/results_{tag}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>